#  Dataset Preprocessing — Waste Dataset Metadata

## 0. Install & Import Libraries

In [ ]:
# Install any missing packages
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['nltk', 'matplotlib', 'seaborn', 'scikit-learn', 'wordcloud']:
    install(pkg)

In [ ]:
import os
import re
import string
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from collections import Counter

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

# Download NLTK resources
for resource in ['stopwords', 'punkt', 'wordnet', 'omw-1.4', 'punkt_tab']:
    nltk.download(resource, quiet=True)

print('All libraries imported successfully.')

## 1. Load & Inspect Data

In [ ]:
CSV_PATH = 'dataset_metadata.csv'   

df = pd.read_csv(CSV_PATH)
print(f' Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(10)

In [ ]:
print('--- DataFrame Info ---')
df.info()

In [ ]:
print('--- Descriptive Statistics ---')
df.describe(include='all')

## 2. Basic Cleaning

In [ ]:
original_shape = df.shape

# --- 2a. Strip leading/trailing whitespace from all string columns ---
str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].apply(lambda col: col.str.strip())

# --- 2b. Normalise label column to lowercase ---
df['label'] = df['label'].str.lower()

# --- 2c. Normalise path separators (Windows → Unix style) ---
df['image_path'] = df['image_path'].str.replace('\\', '/', regex=False)

# --- 2d. Drop exact duplicate rows ---
n_dups = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)

print(f'Original shape : {original_shape}')
print(f'Duplicate rows removed: {n_dups}')
print(f'Shape after cleaning : {df.shape}')

## 3. Missing Value Analysis & Handling

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df)

# Visualise
fig, ax = plt.subplots(figsize=(7, 3))
sns.barplot(x=missing_df.index, y='Missing %', data=missing_df, palette='Reds_d', ax=ax)
ax.set_title('Missing Values per Column (%)', fontsize=13, fontweight='bold')
ax.set_ylabel('%')
plt.tight_layout()
plt.show()

# Handle missing values
df['text_description'] = df['text_description'].fillna('unknown')
df['label'] = df['label'].fillna('unknown')
df = df.dropna(subset=['image_path'])   # path must exist
df = df.reset_index(drop=True)
print(f'\n✅ Missing values handled. Final shape: {df.shape}')

## 4. Feature Engineering

In [ ]:
# --- Extract filename and class folder from image_path ---
df['filename']    = df['image_path'].apply(lambda p: os.path.basename(p))
df['class_folder'] = df['image_path'].apply(lambda p: p.split('/')[-2] if '/' in p else 'unknown')

# --- Text-based features ---
df['text_word_count']  = df['text_description'].apply(lambda x: len(str(x).split()))
df['text_char_count']  = df['text_description'].apply(lambda x: len(str(x)))
df['text_unique_words'] = df['text_description'].apply(lambda x: len(set(str(x).lower().split())))

print('New features added:')
print(df[['filename', 'class_folder', 'text_word_count', 'text_char_count', 'text_unique_words']].head(8))

## 5. Text Preprocessing (NLP)

In [ ]:
STOPWORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer    = PorterStemmer()

def clean_text(text: str) -> str:
    """Full NLP cleaning pipeline."""
    text = str(text).lower()                                   # lowercase
    text = re.sub(r'http\S+|www\.\S+', '', text)              # remove URLs
    text = re.sub(r'\d+', '', text)                           # remove digits
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()                  # collapse whitespace
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]  # remove stopwords
    tokens = [lemmatizer.lemmatize(t) for t in tokens]        # lemmatize
    return ' '.join(tokens)

df['text_clean']   = df['text_description'].apply(clean_text)
df['text_stemmed'] = df['text_description'].apply(
    lambda x: ' '.join([stemmer.stem(t) for t in clean_text(x).split()])
)

print('Sample cleaned text:')
comparison = df[['text_description', 'text_clean', 'text_stemmed']].head(5)
pd.set_option('display.max_colwidth', 80)
comparison

In [ ]:
# --- Identify & flag empty texts after cleaning ---
empty_after_clean = (df['text_clean'].str.strip() == '').sum()
print(f'Rows with empty text after cleaning: {empty_after_clean}')
if empty_after_clean > 0:
    df.loc[df['text_clean'].str.strip() == '', 'text_clean'] = 'no description'
    print('  → Filled with "no description"')

## 6. Label Encoding

In [ ]:
# --- Label Encoder (integer IDs) ---
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])

label_map = dict(zip(le.classes_, le.transform(le.classes_)))
print('Label → Integer mapping:')
for k, v in sorted(label_map.items(), key=lambda x: x[1]):
    print(f'  {k:12s} → {v}')

# --- One-Hot Encoding ---
ohe_df = pd.get_dummies(df['label'], prefix='label').astype(int)
df = pd.concat([df, ohe_df], axis=1)

print(f'\nOHE columns added: {list(ohe_df.columns)}')

## 7. Train / Validation / Test Split

In [ ]:
SEED = 42

# Stratify by label to preserve class distribution
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=SEED, stratify=df['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df['label']
)

# Reset indices
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'Train : {len(train_df):>5,} rows ({len(train_df)/len(df)*100:.1f}%)')
print(f'Val   : {len(val_df):>5,} rows ({len(val_df)/len(df)*100:.1f}%)')
print(f'Test  : {len(test_df):>5,} rows ({len(test_df)/len(df)*100:.1f}%)')

# Verify stratification
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, split, title in zip(axes,
                             [train_df, val_df, test_df],
                             ['Train', 'Validation', 'Test']):
    split['label'].value_counts().plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'{title} — Class Distribution', fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 8. Exploratory Data Analysis (EDA)

In [ ]:
# --- 8a. Class distribution ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

counts = df['label'].value_counts()
axes[0].bar(counts.index, counts.values,
            color=sns.color_palette('Set2', len(counts)), edgecolor='white')
axes[0].set_title('Class Distribution (Count)', fontweight='bold')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

axes[1].pie(counts.values, labels=counts.index,
            autopct='%1.1f%%', startangle=140,
            colors=sns.color_palette('Set2', len(counts)))
axes[1].set_title('Class Distribution (%)', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# --- 8b. Text length distributions ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, title in zip(axes,
                           ['text_word_count', 'text_char_count', 'text_unique_words'],
                           ['Word Count', 'Char Count', 'Unique Words']):
    df[col].hist(bins=20, ax=ax, color='cornflowerblue', edgecolor='white')
    ax.axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean={df[col].mean():.1f}')
    ax.set_title(f'Text — {title}', fontweight='bold')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- 8c. Word count per label (boxplot) ---
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x='label', y='text_word_count',
            palette='Set3', ax=ax)
ax.set_title('Word Count Distribution per Label', fontweight='bold')
ax.set_xlabel('Label')
ax.set_ylabel('Word Count')
plt.tight_layout()
plt.show()

In [ ]:
# --- 8d. Most frequent words (overall) ---
all_words = ' '.join(df['text_clean']).split()
top_words = Counter(all_words).most_common(20)

words, freqs = zip(*top_words)
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(words, freqs, color='mediumpurple', edgecolor='white')
ax.set_title('Top 20 Most Frequent Words (after cleaning)', fontweight='bold')
ax.set_ylabel('Frequency')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# --- 8e. Top words per label ---
labels = df['label'].unique()
n_cols = 3
n_rows = int(np.ceil(len(labels) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for idx, lbl in enumerate(sorted(labels)):
    subset = df[df['label'] == lbl]['text_clean']
    words_lbl = ' '.join(subset).split()
    top = Counter(words_lbl).most_common(10)
    if top:
        w, f = zip(*top)
        axes[idx].barh(w[::-1], f[::-1], color='teal')
        axes[idx].set_title(f'Top words — {lbl}', fontweight='bold')
        axes[idx].set_xlabel('Frequency')

for ax in axes[len(labels):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# --- 8f. Correlation heatmap (numeric features) ---
numeric_cols = ['text_word_count', 'text_char_count', 'text_unique_words', 'label_encoded']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
            linewidths=0.5, square=True)
ax.set_title('Correlation Heatmap — Numeric Features', fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Final Dataset Summary

In [ ]:
print('='*55)
print('  FINAL PREPROCESSED DATASET SUMMARY')
print('='*55)
print(f'  Total samples      : {len(df):,}')
print(f'  Total columns      : {df.shape[1]}')
print(f'  Missing values     : {df.isnull().sum().sum()}')
print(f'  Duplicate rows     : {df.duplicated().sum()}')
print(f'  Classes            : {sorted(df["label"].unique())}')
print(f'  Train/Val/Test     : {len(train_df)} / {len(val_df)} / {len(test_df)}')
print('='*55)
print('\nFinal columns:')
for c in df.columns:
    print(f'  {c}')

df.head()

## 10. Save Processed Data

In [ ]:
os.makedirs('processed', exist_ok=True)

df.to_csv('processed/dataset_full_processed.csv',  index=False)
train_df.to_csv('processed/train.csv',  index=False)
val_df.to_csv('processed/val.csv',    index=False)
test_df.to_csv('processed/test.csv',   index=False)

print('✅ Files saved to ./processed/')
print('   ├── dataset_full_processed.csv')
print('   ├── train.csv')
print('   ├── val.csv')
print('   └── test.csv')